In [ ]:
import torch
import numpy as np
import rasterio
from skimage.exposure import rescale_intensity


def normalize_band(band: np.ndarray) -> np.ndarray:
    """
    Normalize a band between 0 and 1.
    """
    return rescale_intensity(band, out_range=(0, 1))


def compute_entropy_block(block: torch.Tensor, epsilon: float = 1e-10) -> torch.Tensor:
    """
    Compute entropy for a 15x15 pixel block.
    """
    hist = torch.histc(block, bins=256, min=0.0, max=1.0)
    probs = hist / hist.sum()
    probs = torch.clamp(probs, min=epsilon)
    entropy = -torch.sum(probs * torch.log(probs))
    return entropy


# File paths
sentinel2_path = "/home/stagiaire/Téléchargements/TE3/S2.tif"
entropy_output_path = "/home/stagiaire/Téléchargements/TE3/entropy.tif"

# Load Sentinel-2 image
with rasterio.open(sentinel2_path) as src:
    meta = src.meta.copy()
    meta.update({
        'height': src.height // 15,
        'width': src.width // 15,
        'count': 1,
        'dtype': 'float32',
        'transform': src.transform * src.transform.scale(15, 15)
    })

    entropy_avg = np.zeros((meta['height'], meta['width']), dtype=np.float32)

    for i in range(1, 13):
        band = src.read(i)
        band_normalized = normalize_band(band)
        band_tensor = torch.tensor(band_normalized, dtype=torch.float32)

        for y in range(0, band_tensor.size(0) - 15 + 1, 15):
            for x in range(0, band_tensor.size(1) - 15 + 1, 15):
                block = band_tensor[y:y+15, x:x+15]
                entropy = compute_entropy_block(block)
                entropy_avg[y // 15, x // 15] += entropy.item()

    entropy_avg /= 12.0

    with rasterio.open(entropy_output_path, 'w', **meta) as dst:
        dst.write(entropy_avg, 1)

print(f"Entropy TIFF file generated: {entropy_output_path}")
